In [9]:
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio as rio
from shapely.geometry import box
from rasterio.features import rasterize
from rasterio.transform import Affine
from tqdm import tqdm
import math, os
from matplotlib import pyplot as plt
from pathlib import Path

In [ ]:
ROOT = Path.cwd().parent
DATA = ROOT / 'data'
RESULTS = ROOT / 'results'
(RESULTS / 'species-range-rasters' / 'birds').mkdir(parents=True, exist_ok=True)

In [2]:
country_raster_1km_ = rio.open(RESULTS / 'country-rasters' / 'country-raster-eck4-1km.tif')

### birds

In [3]:
birds = gpd.read_file(DATA / 'species-range-maps' / 'BOTW_2024_2.gpkg')

c:\Users\Administrator\.conda\envs\wty\lib\site-packages\pyogrio\geopandas.py:265: UserWarning: More than one layer found in 'BOTW_2024_2.gpkg': 'all_species' (default), 'main_BL_HBW_Checklist_V9'. Specify layer parameter to avoid this warning.
  result = read_func(


In [4]:
birds['seasonal'].unique()

array([2, 4, 3, 1, 5], dtype=int32)

In [5]:
len(birds)

17379

In [6]:
birds_ = birds[(birds['presence'] == (1 or 2 or 3)) & (birds['origin'] == (1 or 2 or 6)) & (birds['seasonal'] == (1 or 2 or 3))]

In [7]:
birds_eck4= birds_.to_crs('ESRI:54012')

In [8]:
birds[(birds['presence'] == (1 or 2 or 3)) & (birds['origin'] == (1 or 2 or 6)) & (birds['seasonal'] == (1 or 2 or 3))]['sisid'].unique().shape

(10374,)

In [10]:
range_index_birds = pd.DataFrame([], columns=['sisid', 'name', 'row_min', 'row_max', 'col_min', 'col_max'])
range_index_birds['sisid'] = birds_eck4['sisid'].unique().tolist()
range_index_birds['name'] = [birds_eck4[birds_eck4['sisid'] == sisid]['sci_name'].values[0] for sisid in range_index_birds['sisid']]

In [11]:
for i in tqdm(range_index_birds.index):
    a_species = range_index_birds.loc[i, 'name']
    try:
        # print(a_species)
        a_species_range = birds_eck4[birds_eck4['sci_name'] == a_species].geometry 
        a_species_polygons = [(a_geometry, 1) for a_geometry in a_species_range]
        reprojected_raster = rasterize(
            shapes=a_species_polygons, 
            out_shape=country_raster_1km_.shape, 
            transform=country_raster_1km_.transform,
            all_touched=True,
            fill=0, 
            dtype=np.uint8,
        )
        rows, cols = np.where(reprojected_raster == 1)
        row_min, row_max = rows.min(), rows.max()
        col_min, col_max = cols.min(), cols.max()
        the_slice = reprojected_raster[row_min:row_max+1, col_min:col_max+1]

        output_path = RESULTS / 'species-range-rasters' / 'birds' / f'{a_species}.tif'
        new_transform = country_raster_1km_.transform * Affine.translation(col_min, row_min)

        with rio.open(
            output_path,
            'w',
            driver='GTiff',
            height=the_slice.shape[0],
            width=the_slice.shape[1],
            count=1,
            dtype=the_slice.dtype, 
            crs=country_raster_1km_.crs,
            transform=new_transform,
            compress='lzw',
        ) as dst:
            dst.write(the_slice, 1)
        
        range_index_birds.loc[i, ['row_min', 'row_max', 'col_min', 'col_max']] = [row_min, row_max, col_min, col_max]
        range_index_birds.to_csv(RESULTS / 'species-range-rasters' / 'range-index-birds.csv', index=False)
    except:
        pass


  0%|          | 0/10374 [00:00<?, ?it/s]

100%|██████████| 10374/10374 [4:33:03<00:00,  1.58s/it]  
